# 02 -- Adding Memories

## What We'll Cover
1. Adding text content
2. Adding URLs (auto-extracted)
3. Uploading files (PDFs, images, videos)
4. Using `customId` for updates and deduplication
5. Scoping with `containerTag` and `containerTags`
6. Bulk ingestion strategies


## 2.1 The `add()` Method

The core ingestion method. Send any raw content and Supermemory extracts memories automatically.

**Parameters:**
- `content` (str): Text, URL, or raw content to process
- `container_tag` (str): Single tag for scoping (user ID, project ID)
- `container_tags` (list[str]): Multiple tags for cross-cutting scopes
- `metadata` (dict): Key-value pairs for filtering
- `custom_id` (str): Your ID to enable updates and prevent duplicates
- `user_id` (str): Partition data to a specific user space


In [ ]:
# 2.2 Adding basic text content
from supermemory import Supermemory
client = Supermemory()

# Simplest form -- just content and a container tag
response = client.add(
    content="Machine learning enables computers to learn from data without explicit programming.",
    container_tag="user_001"
)
print(f"ID: {response.id}, Status: {response.status}")


In [ ]:
# 2.3 Adding conversation transcripts (the primary use case)
conversation = [
    {"role": "assistant", "content": "Hello! How can I help you today?"},
    {"role": "user", "content": "Hi! I am Sarah. I am a product designer at Figma. I love Figma, design systems, and coffee."},
    {"role": "user", "content": "Can you help me design a dashboard layout?"},
]

# Join the conversation into a single text block
conv_text = "\n".join(f"{m['role']}: {m['content']}" for m in conversation)
print("Sending conversation:")
print(conv_text)

response = client.add(
    content=conv_text,
    container_tag="user_sarah",
    metadata={"type": "conversation", "topic": "design"}
)
print(f"\nStored! ID: {response.id}, Status: {response.status}")


In [ ]:
# 2.4 Adding URLs -- Supermemory auto-extracts page content
response = client.add(
    content="https://supermemory.ai/docs/introduction",
    container_tag="knowledge_base",
    metadata={"source": "web", "category": "documentation"}
)
print(f"URL queued for extraction: {response.id}")
print("Note: URL extraction happens asynchronously. Content will be searchable shortly.")


In [ ]:
# 2.5 Adding with containerTags -- multiple scoping dimensions
# containerTags lets you scope a memory to multiple containers
# e.g., a memory belongs to user_001 AND project_alpha AND team_engineering
response = client.memory.create(
    content="The authentication service needs to support OAuth2 and SAML for enterprise customers.",
    containerTags=["user_001", "project_auth", "team_engineering"],
    metadata={"priority": "high", "status": "planned"}
)
print(f"Multi-tagged memory: {response.id}")
print("This memory is now scoped to: user_001, project_auth, team_engineering")


In [ ]:
# 2.6 Using customId -- prevent duplicates, enable updates
# customId is YOUR identifier (conversation ID, document ID, etc.)
# Sending with the same customId updates the existing memory

# First addition
r1 = client.add(
    content="user: Hi, I am Sarah.\nassistant: Nice to meet you, Sarah!",
    custom_id="conv_001",
    container_tag="user_sarah"
)
print(f"First add: {r1.id}")

# Later: add more to the same conversation (same customId)
# Supermemory links these together intelligently
r2 = client.add(
    content="user: What is the weather today?\nassistant: It is sunny and 72 degrees!",
    custom_id="conv_001",  # Same ID -- Supermemory links them
    container_tag="user_sarah"
)
print(f"Update add: {r2.id}")
print("Both fragments are now linked under conv_001")


In [ ]:
# 2.7 Uploading files (PDFs, images, videos)
from pathlib import Path

# Upload a PDF (extracts text + builds memories)
# client.documents.upload_file(
#     file=Path("/path/to/report.pdf"),
#     container_tags=["user_001"],
#     metadata={"type": "report"}
# )

# Supported formats:
# - PDF: Text extraction + chunking + embedding
# - Images: OCR for text in images
# - Videos: Transcription
# - Code: AST-aware chunking

print("File uploads support: PDF, images (OCR), video (transcription), code files")
print("Processing times: PDF ~1-2 min, video ~5-10 min for 100 pages")


In [ ]:
# 2.8 Bulk ingestion -- add multiple items efficiently
contents = [
    ("Alice is a backend engineer specializing in Go and Kubernetes.", "user_alice"),
    ("Bob is a frontend engineer who loves React and design systems.", "user_bob"),
    ("Carol is a data scientist working with PyTorch and JAX.", "user_carol"),
    ("The team uses AWS for infrastructure and GitHub Actions for CI/CD.", "team_eng"),
    ("Sprint 42 focuses on migrating the monolith to microservices.", "sprint_42"),
]

results = []
for content, tag in contents:
    r = client.add(content=content, container_tag=tag)
    results.append(r)
    print(f"Added [{tag}]: {r.id}")

print(f"\nTotal ingested: {len(results)} documents")
print("Tip: For very large batches, use AsyncSupermemory (see Notebook 07)")


## 2.9 Content Types That Work

| Type | Example | Processing |
|------|---------|------------|
| Text | Free-form text, notes | Direct extraction |
| Conversations | Chat transcripts | Role-aware chunking |
| URLs | Web pages | Full-content extraction + indexing |
| PDFs | Documents, reports | Text extraction + chunking |
| Images | Screenshots, photos | OCR for embedded text |
| Videos | Recordings, tutorials | Transcription |
| Code | Source files | AST-aware semantic chunking |

## 2.10 Key Takeaways

- Use `container_tag` for single-scope, `containerTags` for multi-scope
- Use `custom_id` to link updates and prevent duplicates
- URLs are auto-extracted -- just pass the URL as content
- File uploads support PDFs, images (OCR), videos (transcription)
- Processing is async -- status starts as `queued`

**Next:** Notebook 03 -- Searching Memories
